In [3]:
import os
import pandas as pd
import numpy as np
import re

In [5]:
base_path = r"C:\Users\fdsa0\OneDrive\Desktop\pj_data\data_set"
file_name = r"견종_생활시뮬레이션_한글견종명추가.csv"

file_path = os.path.join(base_path, file_name)
df = pd.read_csv(file_path)

In [7]:
print(df.shape)
print(df.columns)

(349, 19)
Index(['breed', 'breed_ko', 'apartment_suitability_1to5',
       'alone_tolerance_1to5', 'barking_tendency_1to5', 'exercise_need_1to5',
       'energy_level_1to5', 'easy_to_groom_1to5', 'shedding_1to5',
       'novice_owner_suitability_1to5', 'breed_group', 'weight_original_lb',
       'min_weight_kg', 'max_weight_kg', 'grooming_frequency_0to1',
       'grooming_frequency_category', 'energy_level_0to1',
       'energy_level_category', 'additional_data_available'],
      dtype='str')


In [8]:
df.isnull().sum()

breed                              0
breed_ko                         144
apartment_suitability_1to5         0
alone_tolerance_1to5               0
barking_tendency_1to5              0
exercise_need_1to5                 0
energy_level_1to5                  0
easy_to_groom_1to5                 0
shedding_1to5                      0
novice_owner_suitability_1to5      0
breed_group                        0
weight_original_lb                 6
min_weight_kg                    144
max_weight_kg                    144
grooming_frequency_0to1          145
grooming_frequency_category      145
energy_level_0to1                143
energy_level_category            143
additional_data_available          0
dtype: int64

In [9]:
drop_columns = [
    "grooming_frequency_0to1",
    "grooming_frequency_category",
    "energy_level_0to1",
    "energy_level_category",
    "additional_data_available"
]

df = df.drop(columns=drop_columns)

df.head()

,breed,breed_ko,apartment_suitability_1to5,alone_tolerance_1to5,barking_tendency_1to5,exercise_need_1to5,energy_level_1to5,easy_to_groom_1to5,shedding_1to5,novice_owner_suitability_1to5,breed_group,weight_original_lb,min_weight_kg,max_weight_kg
0,Afador,NaN,1,3,4,4,4,2,4,1,Mixed Breed Dogs,50 to 75 pounds,NaN,NaN
1,Affenhuahua,NaN,4,1,4,3,4,4,2,4,Mixed Breed Dogs,4 to 12 pounds,NaN,NaN
2,Affenpinscher,아펜핀셔,5,1,2,3,4,3,1,4,Companion Dogs,7 to 9 pounds,3.175147,4.535924
3,Afghan Hound,아프간 하운드,5,2,2,4,5,1,4,3,Hound Dogs,50 to 60 pounds,22.679619,27.215542
4,Airedale Terrier,에어데일 테리어,1,2,4,5,5,2,2,2,Terrier Dogs,40 to 65 pounds,22.679619,31.751466


In [10]:
def parse_weight(weight):
    if pd.isna(weight):
        return np.nan, np.nan

    weight = str(weight).lower().strip()

    # 숫자만 추출
    numbers = re.findall(r"\d+(?:\.\d+)?", weight)
    numbers = [float(n) for n in numbers]

    if len(numbers) == 0:
        return np.nan, np.nan

    # 예: 10 to 20 pounds / 10 - 20 pounds
    if len(numbers) >= 2:
        min_lb = numbers[0]
        max_lb = numbers[1]

    # 예: Up to 12 pounds
    elif "up to" in weight:
        min_lb = np.nan
        max_lb = numbers[0]

    # 예: Starts at 30 pounds
    elif "starts at" in weight or "from" in weight:
        min_lb = numbers[0]
        max_lb = np.nan

    else:
        min_lb = numbers[0]
        max_lb = numbers[0]

    return min_lb, max_lb

In [11]:
df[["parsed_min_lb", "parsed_max_lb"]] = df["weight_original_lb"].apply(
    lambda x: pd.Series(parse_weight(x))
)

df[[
    "breed",
    "weight_original_lb",
    "parsed_min_lb",
    "parsed_max_lb"
]].head(10)

,breed,weight_original_lb,parsed_min_lb,parsed_max_lb
0,Afador,50 to 75 pounds,50.0,75.0
1,Affenhuahua,4 to 12 pounds,4.0,12.0
2,Affenpinscher,7 to 9 pounds,7.0,9.0
3,Afghan Hound,50 to 60 pounds,50.0,60.0
4,Airedale Terrier,40 to 65 pounds,40.0,65.0
5,Akbash,75 to 140 pounds,75.0,140.0
6,Akita,70 to 130 pounds,70.0,130.0
7,Alaskan Klee Kai,10 to 15 pounds,10.0,15.0
8,Alaskan Malamute,75 to 100 pounds,75.0,100.0
9,American Bulldog,60 to 120 pounds,60.0,120.0


In [12]:
LB_TO_KG = 0.453592

df["parsed_min_kg"] = df["parsed_min_lb"] * LB_TO_KG
df["parsed_max_kg"] = df["parsed_max_lb"] * LB_TO_KG

In [13]:
df["min_weight_kg"] = df["min_weight_kg"].fillna(df["parsed_min_kg"])
df["max_weight_kg"] = df["max_weight_kg"].fillna(df["parsed_max_kg"])

In [14]:
df["min_weight_kg"] = df["min_weight_kg"].round(1)
df["max_weight_kg"] = df["max_weight_kg"].round(1)

In [15]:
df = df.drop(columns=[
    "parsed_min_lb",
    "parsed_max_lb",
    "parsed_min_kg",
    "parsed_max_kg"
])

In [16]:
df["avg_weight_kg"] = df[
    ["min_weight_kg", "max_weight_kg"]
].mean(axis=1).round(1)

In [17]:
df[[
    "breed",
    "min_weight_kg",
    "max_weight_kg",
    "avg_weight_kg"
]].head()

,breed,min_weight_kg,max_weight_kg,avg_weight_kg
0,Afador,22.7,34.0,28.4
1,Affenhuahua,1.8,5.4,3.6
2,Affenpinscher,3.2,4.5,3.8
3,Afghan Hound,22.7,27.2,25.0
4,Airedale Terrier,22.7,31.8,27.2


In [18]:
def classify_size(weight):
    if pd.isna(weight):
        return np.nan
    elif weight < 10:
        return "소형견"
    elif weight < 25:
        return "중형견"
    else:
        return "대형견"


df["size"] = df["avg_weight_kg"].apply(classify_size)

In [20]:
df["size"].value_counts(dropna=False)

size
대형견    130
중형견    117
소형견    101
NaN      1
Name: count, dtype: int64

In [21]:
df[[
    "breed",
    "breed_ko",
    "avg_weight_kg",
    "size"
]].head(20)

,breed,breed_ko,avg_weight_kg,size
0,Afador,NaN,28.4,대형견
1,Affenhuahua,NaN,3.6,소형견
2,Affenpinscher,아펜핀셔,3.8,소형견
3,Afghan Hound,아프간 하운드,25.0,대형견
4,Airedale Terrier,에어데일 테리어,27.2,대형견
5,Akbash,NaN,48.8,대형견
6,Akita,아키타,45.4,대형견
7,Alaskan Klee Kai,알래스칸 클리카이,5.6,소형견
8,Alaskan Malamute,알래스칸 말라뮤트,36.3,대형견
9,American Bulldog,아메리칸 불도그,36.3,대형견


In [22]:
df["breed_ko"].isnull().sum()

np.int64(144)

In [23]:
df.isnull().sum()

breed                              0
breed_ko                         144
apartment_suitability_1to5         0
alone_tolerance_1to5               0
barking_tendency_1to5              0
exercise_need_1to5                 0
energy_level_1to5                  0
easy_to_groom_1to5                 0
shedding_1to5                      0
novice_owner_suitability_1to5      0
breed_group                        0
weight_original_lb                 6
min_weight_kg                      1
max_weight_kg                      2
avg_weight_kg                      1
size                               1
dtype: int64

In [24]:
df[df["avg_weight_kg"].isnull()][
    ["breed", "breed_ko", "weight_original_lb"]
]

,breed,breed_ko,weight_original_lb
250,Mutt,NaN,NaN


In [25]:
column_order = [
    "breed",
    "breed_ko",
    "breed_group",
    "weight_original_lb",
    "min_weight_kg",
    "max_weight_kg",
    "avg_weight_kg",
    "size",
    "apartment_suitability_1to5",
    "alone_tolerance_1to5",
    "barking_tendency_1to5",
    "exercise_need_1to5",
    "energy_level_1to5",
    "easy_to_groom_1to5",
    "shedding_1to5",
    "novice_owner_suitability_1to5"
]

df = df[column_order]

In [26]:
df.head()

,breed,breed_ko,breed_group,weight_original_lb,min_weight_kg,max_weight_kg,avg_weight_kg,size,apartment_suitability_1to5,alone_tolerance_1to5,barking_tendency_1to5,exercise_need_1to5,energy_level_1to5,easy_to_groom_1to5,shedding_1to5,novice_owner_suitability_1to5
0,Afador,NaN,Mixed Breed Dogs,50 to 75 pounds,22.7,34.0,28.4,대형견,1,3,4,4,4,2,4,1
1,Affenhuahua,NaN,Mixed Breed Dogs,4 to 12 pounds,1.8,5.4,3.6,소형견,4,1,4,3,4,4,2,4
2,Affenpinscher,아펜핀셔,Companion Dogs,7 to 9 pounds,3.2,4.5,3.8,소형견,5,1,2,3,4,3,1,4
3,Afghan Hound,아프간 하운드,Hound Dogs,50 to 60 pounds,22.7,27.2,25.0,대형견,5,2,2,4,5,1,4,3
4,Airedale Terrier,에어데일 테리어,Terrier Dogs,40 to 65 pounds,22.7,31.8,27.2,대형견,1,2,4,5,5,2,2,2


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 349 entries, 0 to 348
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   breed                          349 non-null    str    
 1   breed_ko                       205 non-null    str    
 2   breed_group                    349 non-null    str    
 3   weight_original_lb             343 non-null    str    
 4   min_weight_kg                  348 non-null    float64
 5   max_weight_kg                  347 non-null    float64
 6   avg_weight_kg                  348 non-null    float64
 7   size                           348 non-null    str    
 8   apartment_suitability_1to5     349 non-null    int64  
 9   alone_tolerance_1to5           349 non-null    int64  
 10  barking_tendency_1to5          349 non-null    int64  
 11  exercise_need_1to5             349 non-null    int64  
 12  energy_level_1to5              349 non-null    int64  
 13  e

In [28]:
df.isnull().sum()

breed                              0
breed_ko                         144
breed_group                        0
weight_original_lb                 6
min_weight_kg                      1
max_weight_kg                      2
avg_weight_kg                      1
size                               1
apartment_suitability_1to5         0
alone_tolerance_1to5               0
barking_tendency_1to5              0
exercise_need_1to5                 0
energy_level_1to5                  0
easy_to_groom_1to5                 0
shedding_1to5                      0
novice_owner_suitability_1to5      0
dtype: int64

In [29]:
df.to_csv(
    "견종_생활시뮬레이션_전처리완료.csv",
    index=False,
    encoding="utf-8-sig"
)

In [30]:
score_cols = [
    "apartment_suitability_1to5",
    "alone_tolerance_1to5",
    "barking_tendency_1to5",
    "exercise_need_1to5",
    "energy_level_1to5",
    "easy_to_groom_1to5",
    "shedding_1to5",
    "novice_owner_suitability_1to5"
]

for col in score_cols:
    print(col, df[col].min(), df[col].max())

apartment_suitability_1to5 1 5
alone_tolerance_1to5 1 5
barking_tendency_1to5 1 5
exercise_need_1to5 1 5
energy_level_1to5 2 5
easy_to_groom_1to5 1 5
shedding_1to5 1 5
novice_owner_suitability_1to5 1 5


In [31]:
df.loc[df["breed"] == "Mutt", "breed_ko"] = "믹스견"
df.loc[df["breed"] == "Mutt", "size"] = "미분류"

In [32]:
save_path = os.path.join(
    base_path,
    "견종_생활시뮬레이션_최종.csv"
)

df.to_csv(save_path, index=False, encoding="utf-8-sig")